# 03 - Market Basket Analysis
Goal: find products that are frequently bought together in UK invoices (FP-Growth), turn them into association rules (support, confidence, lift), remove trivial rules, and translate the strongest ones into cross-sell recommendations.

## 1. Load data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from mlxtend.frequent_patterns import fpgrowth, association_rules

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 60)

PROJECT_DIR = Path.cwd().parent
df = pd.read_parquet(PROJECT_DIR / "data" / "processed"/ "online_retail_clean.parquet")

uk = df[df["Country"] == "United Kingdom"]
print("UK lines:   ", f"{len(uk):,}")
print("UK invoices:", f"{uk['Invoice'].nunique():,}")
print("UK products:", f"{uk['StockCode'].nunique():,}")

UK lines:    918,363
UK invoices: 35,959
UK products: 4,706


## 2. Baskets

In [2]:
size = uk.groupby("Invoice")["StockCode"].nunique()

print(size.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1))
print("\nSingle-product baskets:", f"{(size == 1).mean():.1%}")
for limit in [100, 200, 300]:
    print(f"Baskets with > {limit} products: {(size > limit).sum():,} ({(size > limit).mean():.2%})")

count    35959.0
mean        25.2
std         43.7
min          1.0
50%         15.0
90%         51.0
95%         74.0
99%        204.0
max       1106.0
Name: StockCode, dtype: float64

Single-product baskets: 8.3%
Baskets with > 100 products: 1,136 (3.16%)
Baskets with > 200 products: 378 (1.05%)
Baskets with > 300 products: 180 (0.50%)


In [3]:
MIN_SUPPORT = 0.01
n_baskets = uk["Invoice"].nunique()

item_support = uk.groupby("StockCode")["Invoice"].nunique() / n_baskets
frequent_items = item_support[item_support >= MIN_SUPPORT].index

print("Products total:         ", len(item_support))
print(f"Products with support >= {MIN_SUPPORT:.0%}:", len(frequent_items))
print("Min baskets needed:     ", int(np.ceil(MIN_SUPPORT * n_baskets)))

Products total:          4706
Products with support >= 1%: 701
Min baskets needed:      360


In [4]:
MAX_ITEMS = 200
keep = size[size <= MAX_ITEMS].index
uk_b = uk[uk["Invoice"].isin(keep)]

print("Baskets removed:", f"{len(size) - len(keep):,} ({1 - len(keep) / len(size):.2%})")
print("Lines removed:  ", f"{1 - len(uk_b) / len(uk):.1%} of UK lines")

# recompute support on the capped baskets
n_baskets = uk_b["Invoice"].nunique()
item_support = uk_b.groupby("StockCode")["Invoice"].nunique() / n_baskets
frequent_items = item_support[item_support >= MIN_SUPPORT].index
print("Baskets:", f"{n_baskets:,}","| Frequent products:", len(frequent_items))

Baskets removed: 378 (1.05%)
Lines removed:   14.2% of UK lines
Baskets: 35,581 | Frequent products: 595


In [5]:
basket = (uk_b[uk_b["StockCode"].isin(frequent_items)]
          .drop_duplicates(["Invoice", "StockCode"])
          .assign(bought = True)
          .pivot(index = "Invoice", columns = "StockCode", values = "bought")
          .notna()
          .reindex(uk_b["Invoice"].unique(), fill_value = False)) # keep All baskets

print("Matrix shape:", basket.shape)
print("Top item support:", round(basket.mean().max(), 3))

Matrix shape: (35581, 595)
Top item support: 0.14


In [6]:
%%time
itemsets = fpgrowth(basket, min_support=MIN_SUPPORT, use_colnames=True)
itemsets["length"] = itemsets["itemsets"].apply(len)
print(itemsets["length"].value_counts().sort_index())

length
1    595
2    327
3     85
4      1
Name: count, dtype: int64
CPU times: total: 8.94 s
Wall time: 8.96 s


In [7]:
names = uk_b.drop_duplicates("StockCode").set_index("StockCode")["Description"]
itemsets["items"] = itemsets["itemsets"].apply(lambda s: " + ".join(names[c] for c in s))

itemsets[itemsets["length"] == 2].nlargest(15, "support")[["support", "items"]]

,support,items
923,0.034906,JUMBO BAG PINK POLKADOT + JUMBO BAG RED RETROSPOT
755,0.031730,JUMBO BAG RED RETROSPOT + JUMBO STORAGE BAG SUKI
648,0.031337,WHITE HANGING HEART T-LIGHT HOLDER + RED HANGING HEART T...
787,0.029510,LUNCH BAG BLACK SKULL. + LUNCH BAG RED RETROSPOT
831,0.029032,WOODEN PICTURE FRAME WHITE FINISH + WOODEN FRAME ANTIQUE...
847,0.028808,JUMBO BAG RED RETROSPOT + JUMBO BAG STRAWBERRY
888,0.028189,LUNCH BAG SUKI DESIGN + LUNCH BAG RED RETROSPOT
942,0.028161,JUMBO BAG RED RETROSPOT + JUMBO SHOPPER VINTAGE RED PAISLEY
905,0.027992,LUNCH BAG RED RETROSPOT + LUNCH BAG PINK POLKADOT
819,0.027262,JUMBO BAG BAROQUE BLACK WHITE + JUMBO BAG RED RETROSPOT


## 5. Association rules

In [8]:
rules = association_rules(itemsets[["support", "itemsets"]],
                          num_itemsets=len(basket),
                          metric="lift", min_threshold=1)

rules["antecedent"] = rules["antecedents"].apply(lambda s: " + ".join(names[c] for c in s))
rules["consequent"] = rules["consequents"].apply(lambda s: " + ".join(names[c] for c in s))

print("Rules with lift > 1:", len(rules))
print(rules[["support", "confidence", "lift"]].describe().round(3))

Rules with lift > 1: 1178
        support  confidence      lift
count  1178.000    1178.000  1178.000
mean      0.013       0.375    10.406
std       0.004       0.182     7.597
min       0.010       0.072     1.305
25%       0.011       0.224     6.373
50%       0.012       0.351     9.033
75%       0.014       0.518    11.322
max       0.035       0.900    55.102


In [9]:
import re
COLOURS = {"RED", "PINK", "BLUE", "WHITE", "BLACK", "GREEN", "IVORY", "CREAM",
           "YELLOW", "ORANGE", "PURPLE", "GOLD", "SILVER", "BROWN", "GREY", "NATURAL"}
STOP = {"OF", "SET", "PACK", "THE", "AND", "WITH", "IN", "DESIGN"} | COLOURS

def words(text):
    return {w for w in re.findall(r"[A-Z]+", text.upper()) if len(w) > 1} - STOP

def similar(a, b, threshold=1/3):
    wa, wb = words(a), words(b)
    if not wa or not wb:
        return False
    return len(wa & wb) / min(len(wa), len(wb)) >= threshold

def same_family_rule(ante, cons):
    # same family if ANY antecedent item is similar to ANY consequent item
    return any(similar(names[a], names[c]) for a in ante for c in cons)

rules["same_family"] = [same_family_rule(a, c)
                        for a, c in zip(rules["antecedents"], rules["consequents"])]

print("Same-family rules: ", rules["same_family"].sum(), f"({rules['same_family'].mean():.0%})")
print("Cross-family rules:", (~rules["same_family"]).sum())

rules.sample(20, random_state=1)[["antecedent", "consequent", "lift", "same_family"]]

Same-family rules:  1038 (88%)
Cross-family rules: 140


,antecedent,consequent,lift,same_family
660,JUMBO BAG BAROQUE BLACK WHITE + JUMBO BAG RED RETROSPOT,JUMBO BAG PINK POLKADOT,8.934512,True
548,SET/6 RED SPOTTY PAPER CUPS,SET/6 RED SPOTTY PAPER PLATES,38.416226,True
1177,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,32.012823,True
321,JUMBO BAG SCANDINAVIAN PAISLEY,JUMBO BAG RED RETROSPOT + JUMBO STORAGE BAG SUKI,9.412815,True
1053,FELTCRAFT CUSHION BUTTERFLY,FELTCRAFT CUSHION OWL,25.431908,True
142,LUNCH BAG CARS BLUE,LUNCH BAG SPACEBOY DESIGN + LUNCH BAG SUKI DESIGN,10.458532,True
672,JUMBO BAG RED RETROSPOT + JUMBO SHOPPER VINTAGE RED PAISLEY,JUMBO BAG BAROQUE BLACK WHITE,9.061023,True
1132,PAPER CHAIN KIT VINTAGE CHRISTMAS,60 CAKE CASES VINTAGE CHRISTMAS,11.590389,True
861,JUMBO BAG RED RETROSPOT,LUNCH BAG SPACEBOY DESIGN,2.330949,True
547,ASSORTED COLOUR BIRD ORNAMENT,PAINTED METAL PEARS ASSORTED,10.156014,False


In [10]:
rules["pair"] = [frozenset(a | c) for a, c in zip(rules["antecedents"], rules["consequents"])]

cross = (rules[~rules["same_family"]]
         .sort_values("confidence", ascending=False)
         .drop_duplicates("pair"))        # one directuon per product group

cols = ["antecedent", "consequent", "support", "confidence", "lift"]
top = cross[cross["confidence"] >= 0.3].nlargest(20, "lift")
print("Cross-family links (unique):", len(cross))
top[cols].round(3)

Cross-family links (unique): 70


,antecedent,consequent,support,confidence,lift
783,RED RETROSPOT SUGAR JAM BOWL,RED RETROSPOT SMALL MILK JUG,0.011,0.639,23.516
884,TOY TIDY PINK POLKADOT,RECYCLING BAG RETROSPOT,0.014,0.550,15.504
537,COOK WITH WINE METAL SIGN,HAND OVER THE CHOCOLATE SIGN,0.014,0.398,10.558
546,PAINTED METAL PEARS ASSORTED,ASSORTED COLOUR BIRD ORNAMENT,0.014,0.732,10.156
518,HAND OVER THE CHOCOLATE SIGN,GIN + TONIC DIET METAL SIGN,0.015,0.393,9.684
541,NO SINGING METAL SIGN,HAND OVER THE CHOCOLATE SIGN,0.011,0.357,9.481
406,WOOD 2 DRAWER CABINET WHITE FINISH,WOODEN PICTURE FRAME WHITE FINISH,0.013,0.425,8.301
402,WOOD 2 DRAWER CABINET WHITE FINISH,WOODEN FRAME ANTIQUE WHITE,0.013,0.434,8.217
514,HAND OVER THE CHOCOLATE SIGN,PLEASE ONE PERSON METAL SIGN,0.014,0.362,7.963
610,JUMBO BAG WOODLAND ANIMALS,JUMBO SHOPPER VINTAGE RED PAISLEY,0.012,0.419,7.646
